In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim, when, current_timestamp, datediff

# 1. KẾT NỐI ĐẾN STORAGE ACCOUNT
spark.conf.set(
    "fs.azure.account.key.dbprojectcovid19.dfs.core.windows.net",
    "my-HashKey"
)

# 2. TẠO ĐƯỜNG DẪN
bronze_path = "abfss://bronze@dbprojectcovid19.dfs.core.windows.net/"
silver_path = "abfss://silver@dbprojectcovid19.dfs.core.windows.net/"


# 3. ĐỌC FOLDER BRONZE
df_orders = spark.read.csv(
    bronze_path + "olist_orders_dataset/",
    header=True,
    inferSchema=True
)
df_products = spark.read.csv(
    bronze_path + "olist_products_dataset/",
    header=True,
    inferSchema=True
)

df_sellers = spark.read.csv(
    bronze_path + "olist_sellers_dataset/",
    header=True,
    inferSchema=True
)

df_customers = spark.read.csv(
    bronze_path + "olist_customers_dataset/",
    header=True,
    inferSchema=True
)

df_geolocation = spark.read.csv(
    bronze_path + "olist_geolocation_dataset/",
    header=True,
    inferSchema=True
)

df_order_items = spark.read.csv(
    bronze_path + "olist_order_items_dataset/",
    header=True,
    inferSchema=True
)

df_order_payments = spark.read.csv(
    bronze_path + "olist_order_payments_dataset/",
    header=True,
    inferSchema=True
)

df_order_reviews = spark.read.csv(
    bronze_path + "olist_order_reviews_dataset/",
    header=True,
    inferSchema=True
)
# HIỂN THỊ MỘT VÀI DỮ LIỆU KIỂM TRA
display(df_orders.limit(5))
display(df_products.limit(5))
display(df_sellers.limit(5))
display(df_customers.limit(5))
display(df_geolocation.limit(5))
display(df_order_items.limit(5))
display(df_order_payments.limit(5))
display(df_order_reviews.limit(5))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33Z,2017-10-02T11:07:15Z,2017-10-04T19:55:00Z,2017-10-10T21:25:13Z,2017-10-18T00:00:00Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37Z,2018-07-26T03:24:27Z,2018-07-26T14:31:00Z,2018-08-07T15:27:45Z,2018-08-13T00:00:00Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49Z,2018-08-08T08:55:23Z,2018-08-08T13:50:00Z,2018-08-17T18:06:29Z,2018-09-04T00:00:00Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06Z,2017-11-18T19:45:59Z,2017-11-22T13:39:59Z,2017-12-02T00:28:42Z,2017-12-15T00:00:00Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39Z,2018-02-13T22:20:29Z,2018-02-14T19:46:34Z,2018-02-16T18:17:02Z,2018-02-26T00:00:00Z


product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [ ]:
# 4. CHUẨN HOÁ DỮ LIỆU VÀ LÀM SẠCH

# ORDERS
df_orders_silver = df_orders.select(
    col("order_id"),
    col("customer_id"),
    lower(trim(col("order_status"))).alias("order_status"),
    col("order_purchase_timestamp").cast("timestamp"),
    col("order_approved_at").cast("timestamp"),
    col("order_delivered_carrier_date").cast("timestamp"),
    col("order_delivered_customer_date").cast("timestamp"),
    col("order_estimated_delivery_date").cast("timestamp")
)

df_orders_silver = df_orders_silver.filter(col("order_id").isNotNull())

df_orders_silver = df_orders_silver.fillna({
    "order_status": "unknown"
})

df_orders_silver = df_orders_silver.filter(
    col("order_purchase_timestamp").isNotNull()
)

df_orders_silver = df_orders_silver.withColumn(
    "delivery_days",
    datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
)

df_orders_silver = df_orders_silver.withColumn(
    "is_valid",
    col("delivery_days") >= 0
)

df_orders_silver = df_orders_silver.dropDuplicates(["order_id"])

df_orders_silver = df_orders_silver.withColumn("processed_at", current_timestamp())


# PRODUCTS
df_products_silver = df_products.select(
    col("product_id"),
    col("product_category_name"),
    col("product_name_lenght"),
    col("product_description_lenght"),
    col("product_photos_qty"),
    col("product_weight_g"),
    col("product_length_cm"),
    col("product_height_cm"),
    col("product_width_cm")
)

df_products_silver = df_products.withColumn(
    "product_category_name",
    when(col("product_category_name").isNull(), "unknown")
    .otherwise(lower(trim(col("product_category_name"))))
)

df_products_silver = df_products_silver.dropDuplicates(["product_id"]) \
    .withColumn("processed_at", current_timestamp())


# SELLERS
df_sellers_silver = df_sellers.select(
    col("seller_id"),
    col("seller_zip_code_prefix"),
    col("seller_city"),
    col("seller_state")
)
df_sellers_silver = df_sellers.withColumn(
    "seller_city",
    lower(trim(col("seller_city")))
)

df_sellers_silver = df_sellers_silver.dropDuplicates(["seller_id"]) \
    .withColumn("processed_at", current_timestamp())

# 4. CUSTOMERS
df_customers_silver = df_customers.select(
    col("customer_id"),
    col("customer_unique_id"),
    col("customer_zip_code_prefix"),
    lower(trim(col("customer_city"))).alias("customer_city"),
    lower(trim(col("customer_state"))).alias("customer_state")
)
df_customers_silver = df_customers_silver.dropDuplicates(["customer_id"]) \
    .withColumn("processed_at", current_timestamp())


# 5. GEOLOCATION
df_geolocation_silver = df_geolocation.select(
    col("geolocation_zip_code_prefix"),
    col("geolocation_lat"),
    col("geolocation_lng"),
    lower(trim(col("geolocation_city"))).alias("geolocation_city"),
    lower(trim(col("geolocation_state"))).alias("geolocation_state")
)
# Bảng này cực kỳ nhiều dữ liệu trùng, dùng distinct để tối ưu dung lượng
df_geolocation_silver = df_geolocation_silver.distinct() \
    .withColumn("processed_at", current_timestamp())


# 6. ORDER ITEMS
df_order_items_silver = df_order_items.select(
    col("order_id"),
    col("order_item_id"),
    col("product_id"),
    col("seller_id"),
    col("shipping_limit_date").cast("timestamp"),
    col("price").cast("double"),
    col("freight_value").cast("double")
)
df_order_items_silver = df_order_items_silver.filter(col("order_id").isNotNull())
df_order_items_silver = df_order_items_silver.dropDuplicates(["order_id", "order_item_id"]) \
    .withColumn("processed_at", current_timestamp())


# 7. ORDER PAYMENTS
df_order_payments_silver = df_order_payments.select(
    col("order_id"),
    col("payment_sequential"),
    lower(trim(col("payment_type"))).alias("payment_type"),
    col("payment_installments").cast("int"),
    col("payment_value").cast("double")
)
# Loại bỏ các giao dịch có giá trị lỗi (nếu có)
df_order_payments_silver = df_order_payments_silver.filter(col("payment_value") >= 0) \
    .withColumn("processed_at", current_timestamp())


#  ORDER REVIEWS
df_order_reviews_silver = df_order_reviews.select(
    col("review_id"),
    col("order_id"),
    col("review_score").cast("int"),
    trim(col("review_comment_title")).alias("review_comment_title"),
    trim(col("review_comment_message")).alias("review_comment_message"),
    col("review_creation_date").cast("timestamp"),
    col("review_answer_timestamp").cast("timestamp")
)
# Xử lý các giá trị Null trong phần comment để tránh lỗi khi phân tích text
df_order_reviews_silver = df_order_reviews_silver.fillna({
    "review_comment_title": "no title",
    "review_comment_message": "no message"
})
df_order_reviews_silver = df_order_reviews_silver.dropDuplicates(["review_id"]) \
    .withColumn("processed_at", current_timestamp())

In [ ]:
# 5. LƯU VÀO FOLDER SILVER DƯỚI DẠNG PARQUET FILE
print("✅ Đang lưu trữ dữ liệu vào Delta Lake...")
df_orders_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "orders")

df_products_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "products")

df_sellers_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "sellers")

df_customers_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "customers")

df_geolocation_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "geolocation")

df_order_items_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "order_items")

df_order_payments_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "order_payments")

df_order_reviews_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "order_reviews")

print("✅ Hoàn thành xử lý và lưu trữ toàn bộ 8 bảng tầng Silver!")



In [ ]:
# XEM DỮ LIỆU Ở TẦNG SILVER
silver_df = spark.read.format("delta").load(silver_path + "orders")
display(silver_df)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,is_valid,processed_at
cedd898e1a953e9a8f3d763a364f6717,d44c356f894fa28cffe33c455abb72ba,delivered,2017-08-17T17:03:31Z,2017-08-17T18:05:30Z,2017-08-18T19:57:56Z,2017-08-25T17:45:17Z,2017-09-13T00:00:00Z,8,true,2026-03-17T16:40:25.239806Z
32bda40452bf0485c310bca24eb53886,ebfb3f2e532c37d5a959ded000c9f603,delivered,2018-07-06T18:16:06Z,2018-07-10T08:45:18Z,2018-07-11T11:51:00Z,2018-07-16T13:11:06Z,2018-07-27T00:00:00Z,10,true,2026-03-17T16:40:25.239806Z
186d50e82bb60fc547f3dbf7aa6f6f8a,ff0a44202beb5310644a0ea439973f4d,delivered,2018-08-19T11:28:11Z,2018-08-20T16:50:16Z,2018-08-21T19:01:00Z,2018-08-27T14:11:55Z,2018-08-28T00:00:00Z,8,true,2026-03-17T16:40:25.239806Z
f6f6a68d78c7dd4b319f6df3c846094b,aa012a8928021b41ea6de61e855880bc,delivered,2018-04-13T12:10:45Z,2018-04-13T13:15:45Z,2018-04-17T18:48:53Z,2018-04-27T11:58:45Z,2018-05-22T00:00:00Z,14,true,2026-03-17T16:40:25.239806Z
f5218a2401f92542f87b1de5bc3f92ff,997070d4812561b1a8ededcbb6c2429e,delivered,2018-03-02T11:04:18Z,2018-03-02T11:15:45Z,2018-03-05T20:55:04Z,2018-03-06T13:38:58Z,2018-03-14T00:00:00Z,4,true,2026-03-17T16:40:25.239806Z
a1ba9164e568e28f6977c343fb78e9b1,72600f002ba1550d6432cb91464f6969,delivered,2018-03-11T22:14:38Z,2018-03-11T22:28:09Z,2018-03-12T17:24:59Z,2018-03-31T14:18:36Z,2018-04-03T00:00:00Z,20,true,2026-03-17T16:40:25.239806Z
4c342c53eed805e5712a130cccf665ee,fa3b701e87d5470f0caa38474274f11f,delivered,2017-10-04T21:45:01Z,2017-10-06T02:25:13Z,2017-10-06T19:28:49Z,2017-10-17T20:47:50Z,2017-11-06T00:00:00Z,13,true,2026-03-17T16:40:25.239806Z
e2eb25bad1122ea7060e7e1b97feea74,ef115244104b1f97ef5710a0441cab4d,delivered,2018-01-15T14:12:49Z,2018-01-15T14:30:41Z,2018-01-19T12:38:54Z,2018-01-30T15:33:22Z,2018-02-14T00:00:00Z,15,true,2026-03-17T16:40:25.239806Z
1431ee47e31c27528b8ae56b66689afc,5e43c982ccb91984104e2aaaa062e652,delivered,2017-08-09T18:20:16Z,2017-08-11T16:15:13Z,2017-08-14T14:35:01Z,2017-08-15T22:29:56Z,2017-08-24T00:00:00Z,6,true,2026-03-17T16:40:25.239806Z
3821dc9ac1b25bed42324d0885b0a930,dd3926dff45a3459c4bdac0b58a2a0bf,delivered,2018-04-17T21:01:58Z,2018-04-17T21:15:10Z,2018-04-18T18:52:45Z,2018-04-23T18:01:57Z,2018-05-09T00:00:00Z,6,true,2026-03-17T16:40:25.239806Z
